# Task Performance
This notebook calculates all the relevant task performance metrics. 

In [1]:
import pandas as pd
import bson
import uuid
import numpy as np
import re
from scipy import stats as st
from statsmodels.stats.multitest import multipletests
import pingouin as pg
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
participants = pd.read_csv("../ueq/participants.csv", encoding="UTF-8", delimiter=";")
with open("../database/chat.bson", "rb") as f:
    all_chats = bson.decode_all(f.read())

with open("../database/messages.bson", "rb") as f:
    all_messages = bson.decode_all(f.read())

print(f'\
        {len(participants["uid"])} participants imported\
        {len(all_chats)} chats imported\
        {len(all_messages)} messages imported')

        20 participants imported        63 chats imported        1236 messages imported


In [3]:
# final df for all task performance metrics
df_metrics = pd.DataFrame([])

df_participants_clean = participants[['ID', 'uid', 'Group']].rename(
    columns={
        'ID': 'pid',
        'Group': 'group'
    }
)

# standardize uids
df_participants_clean['uid'] = df_participants_clean['uid'].astype(str).str.strip().str.lower()
df_participants_clean

,pid,uid,group
0,1,3d3fad0c-f4e2-491f-97b3-95c71b7c6723,control_first
1,2,700afe5f-e6dc-4191-9ffc-2d540d21780d,experiment_first
2,3,071b323f-15a1-4f26-8510-9a857540cf04,experiment_first
3,4,ecd6e48c-d160-4ad1-b576-772aee78a77a,control_first
4,5,f21ef8da-e2e6-4a07-b757-2c2b8caa13df,experiment_first
5,6,2dbd9cc6-7d1f-4b50-9dc4-f22ba6c4aa09,control_first
6,7,4ddbf84b-633f-49e1-9412-e52629d7eb1e,experiment_first
7,8,28ea6a5b-7ea9-41b2-851c-1317b0df0019,experiment_first
8,9,05e5bd34-f2d8-4a3d-9453-e9f413cdfcf7,control_first
9,10,cd83ed7e-57d0-4ec5-8210-5acbbe5accae,control_first


### Task Success Rate
We calculate the TSR with this formula: Completed tasks / 7 * 100
We calculate the TSR for each user and both chat conditions. After the TSR was caluclated we will evaluate if the results are significant. 

In [4]:
# All chats have a task list length of 7 
TOTAL_TASK_COUNT = 7
conditions = {"control", "experiment"}

# first we filter out all chats that don't belong to a valid participant account
def filter_chats(_all_chats: list[dict[str, any]]):
    valid_chats = []

    for chat in _all_chats:
        chat_uid = chat.get("uid")
        chat_condition = chat.get("condition")

        # select only the chats from valid uids
        # ignore warmup chat
        if chat_uid in participants["uid"].values and chat_condition in conditions:
            chat["_id"] = str(chat.get("_id"))
            valid_chats.append(chat)
            print(f"uid {chat_uid} is in participants list.")

        else: 
            continue

    valid_chats_len = len(valid_chats)
    print(f"{valid_chats_len} valid chats found.\
          {"Chat length is VALID" if valid_chats_len == 40 else "Chat length is INVALID"}")

    return valid_chats

filtered_chats: list[dict[str, any]] = filter_chats(all_chats)

assert(
    len(filtered_chats) == 40
), "Error chat length is INCORRECT"


uid 3d3fad0c-f4e2-491f-97b3-95c71b7c6723 is in participants list.
uid 3d3fad0c-f4e2-491f-97b3-95c71b7c6723 is in participants list.
uid 700afe5f-e6dc-4191-9ffc-2d540d21780d is in participants list.
uid 700afe5f-e6dc-4191-9ffc-2d540d21780d is in participants list.
uid 071b323f-15a1-4f26-8510-9a857540cf04 is in participants list.
uid 071b323f-15a1-4f26-8510-9a857540cf04 is in participants list.
uid ecd6e48c-d160-4ad1-b576-772aee78a77a is in participants list.
uid ecd6e48c-d160-4ad1-b576-772aee78a77a is in participants list.
uid f21ef8da-e2e6-4a07-b757-2c2b8caa13df is in participants list.
uid f21ef8da-e2e6-4a07-b757-2c2b8caa13df is in participants list.
uid 2dbd9cc6-7d1f-4b50-9dc4-f22ba6c4aa09 is in participants list.
uid 2dbd9cc6-7d1f-4b50-9dc4-f22ba6c4aa09 is in participants list.
uid 4ddbf84b-633f-49e1-9412-e52629d7eb1e is in participants list.
uid 4ddbf84b-633f-49e1-9412-e52629d7eb1e is in participants list.
uid 28ea6a5b-7ea9-41b2-851c-1317b0df0019 is in participants list.
uid 28ea6a

In [5]:
tsr_metrics = []

for chat in filtered_chats:
    condition = chat.get("condition")
    
    # process only control and experiment chats
    uid = str(chat.get("uid"))
    msg_chat_id = chat.get("_id")
    task_list = chat.get("taskList", [])
    
    # count the number of successfully completed tasks
    completed_tasks_count = 0
    if isinstance(task_list, list):
        for task in task_list:
            if task.get("completed") is True:
                completed_tasks_count += 1
    
    # calculate Task Success Rate (TSR) as a percentage
    tsr_percentage = (completed_tasks_count / TOTAL_TASK_COUNT) * 100
    
    tsr_metrics.append({
        "uid": uid,
        "chat_id": msg_chat_id,
        "condition": condition,
        "completed_tasks": completed_tasks_count,
        "total_tasks": TOTAL_TASK_COUNT,
        "tsr": tsr_percentage
    })

# create df
df_tsr_metrics = pd.DataFrame(tsr_metrics)
df_metrics = df_tsr_metrics.merge(df_participants_clean, on='uid', how='inner')

# merge the participants and tsr metrics to create a master metrics df
# also reorder the columns
column_order = ['pid', 'uid', 'group', 'chat_id', 'condition', 'completed_tasks', 'total_tasks', 'tsr']
df_metrics = df_metrics[column_order]
df_metrics


,pid,uid,group,chat_id,condition,completed_tasks,total_tasks,tsr
0,1,3d3fad0c-f4e2-491f-97b3-95c71b7c6723,control_first,6a171aff685d7bc79ba36719,control,7,7,100.000000
1,1,3d3fad0c-f4e2-491f-97b3-95c71b7c6723,control_first,6a171ef3685d7bc79ba3674b,experiment,7,7,100.000000
2,2,700afe5f-e6dc-4191-9ffc-2d540d21780d,experiment_first,6a18856a36467a7b3556b83d,experiment,7,7,100.000000
3,2,700afe5f-e6dc-4191-9ffc-2d540d21780d,experiment_first,6a18885136467a7b3556b877,control,7,7,100.000000
4,3,071b323f-15a1-4f26-8510-9a857540cf04,experiment_first,6a195c4160e5926604688f08,experiment,5,7,71.428571
5,3,071b323f-15a1-4f26-8510-9a857540cf04,experiment_first,6a195e8c60e5926604688f4c,control,5,7,71.428571
6,4,ecd6e48c-d160-4ad1-b576-772aee78a77a,control_first,6a1d38207f563a2d5e13bb09,control,7,7,100.000000
7,4,ecd6e48c-d160-4ad1-b576-772aee78a77a,control_first,6a1d3baa7f563a2d5e13bb3f,experiment,7,7,100.000000
8,5,f21ef8da-e2e6-4a07-b757-2c2b8caa13df,experiment_first,6a212980ffef44b6da07364f,experiment,7,7,100.000000
9,5,f21ef8da-e2e6-4a07-b757-2c2b8caa13df,experiment_first,6a212b8fffef44b6da07367b,control,7,7,100.000000


## Metric 2: Conversational Verbosity / Average Message Length (AML)

### Theoretical Construct
Average Message Length (AML) serves as a proxy metric captured at the intersection of **User Engagement** and **Linguistic Friction**. 

In conversational AI and language-learning studies, a higher word count per message can be interpreted through two distinct lenses:
1. **High User Engagement:** The participant is highly motivated, elaborates extensively, and engages deeply in the foreign language context.
2. **Linguistic Friction:** The system fails to understand the user's intent, forcing the participant to write longer, repetitive, or structurally complex prompts to rectify AI misconceptions (to be evaluated qualitatively via chat transcripts later).

### Mathematical Definition
For each participant session $i$ within a specific experimental condition, AML is defined as the total number of words typed by the user divided by the total number of distinct messages sent by the user:

$$\text{AML}_{i} = \frac{\sum(\text{Words in all human user messages in session } i)}{\text{Total count of human user messages in session } i}$$

*Note: AI companion responses (`isUser: false`) are strictly excluded from both the numerator and the denominator to reflect only the user's linguistic behavior.*

In [6]:
# now we filter out all messages that don't belong to a valid chat 
def filter_messages(_all_messages: list[dict[str, any]]):
    valid_messages = []

    # we need to convert the chat_id to string because it has the ObjectId datatype from MongoDB
    valid_chat_ids_set = set(df_metrics["chat_id"].astype(str).values)

    for msg in _all_messages:
        msg_id = msg.get("_id")
        msg_chat_id = str(msg.get("chatId"))

        print(msg_chat_id)

        # select only the chats from valid uids
        # ignore warmup chat
        if msg_chat_id in valid_chat_ids_set:
            valid_messages.append(msg)
            print(f"message {msg_id} from chat {msg_chat_id} is valid.")

        else: 
            continue

    valid_messages_len = len(valid_messages)
    print(f"{valid_messages_len} valid messages found.")

    return valid_messages
    
filtered_messages: list[dict[str, any]] = filter_messages(all_messages)

6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a1712e8685d7bc79ba366c9
6a171aff685d7bc79ba36719
message 6a171b5f685d7bc79ba3671e from chat 6a171aff685d7bc79ba36719 is valid.
6a171aff685d7bc79ba36719
message 6a171b61685d7bc79ba3671f from chat 6a171aff685d7bc79ba36719 is valid.
6a171aff685d7bc79ba36719
message 6a171ba0685d7bc79ba36722 from chat 6a171aff685d7bc79ba36719 is valid.
6a171aff685d7bc79ba36719
message 6a171ba2685d7bc79ba36723 from chat 6a171aff685d7bc79ba36719 is valid.
6a171aff685d7bc79ba36719
message 6a171

In [7]:
chat_message_stats = {}

for msg in filtered_messages:
    # Filter: Strictly keep only messages sent by the human user
    if msg.get("isUser") is True:
        msg_chat_id = str(msg.get("chatId"))
        text = msg.get("text", "")
        
        if not isinstance(text, str):
            text = ""
            
        # Clean whitespaces and split into words
        words = [w for w in re.split(r'\s+', text.strip()) if w]
        word_count = len(words)
        
        if msg_chat_id not in chat_message_stats:
            chat_message_stats[msg_chat_id] = {
                "total_user_words": 0,
                "total_user_messages": 0
            }
            
        chat_message_stats[msg_chat_id]["total_user_words"] += word_count
        chat_message_stats[msg_chat_id]["total_user_messages"] += 1

# Calculate Average Message Length (AML) for each session
aml_data = []
for msg_chat_id, stats in chat_message_stats.items():
    total_words = stats["total_user_words"]
    total_msgs = stats["total_user_messages"]
    
    # Prevent division by zero if a chat has no user messages
    aml = (total_words / total_msgs) if total_msgs > 0 else 0.0
    
    aml_data.append({
        "chat_id": msg_chat_id,
        "total_user_words": total_words,
        "total_user_messages": total_msgs,
        "aml": aml
    })

df_aml = pd.DataFrame(aml_data)
df_metrics = df_metrics.merge(df_aml, on="chat_id", how="left")
df_metrics


,pid,uid,group,chat_id,condition,completed_tasks,total_tasks,tsr,total_user_words,total_user_messages,aml
0,1,3d3fad0c-f4e2-491f-97b3-95c71b7c6723,control_first,6a171aff685d7bc79ba36719,control,7,7,100.000000,43,9,4.777778
1,1,3d3fad0c-f4e2-491f-97b3-95c71b7c6723,control_first,6a171ef3685d7bc79ba3674b,experiment,7,7,100.000000,44,8,5.500000
2,2,700afe5f-e6dc-4191-9ffc-2d540d21780d,experiment_first,6a18856a36467a7b3556b83d,experiment,7,7,100.000000,54,9,6.000000
3,2,700afe5f-e6dc-4191-9ffc-2d540d21780d,experiment_first,6a18885136467a7b3556b877,control,7,7,100.000000,29,8,3.625000
4,3,071b323f-15a1-4f26-8510-9a857540cf04,experiment_first,6a195c4160e5926604688f08,experiment,5,7,71.428571,60,14,4.285714
5,3,071b323f-15a1-4f26-8510-9a857540cf04,experiment_first,6a195e8c60e5926604688f4c,control,5,7,71.428571,59,14,4.214286
6,4,ecd6e48c-d160-4ad1-b576-772aee78a77a,control_first,6a1d38207f563a2d5e13bb09,control,7,7,100.000000,49,7,7.000000
7,4,ecd6e48c-d160-4ad1-b576-772aee78a77a,control_first,6a1d3baa7f563a2d5e13bb3f,experiment,7,7,100.000000,43,11,3.909091
8,5,f21ef8da-e2e6-4a07-b757-2c2b8caa13df,experiment_first,6a212980ffef44b6da07364f,experiment,7,7,100.000000,59,9,6.555556
9,5,f21ef8da-e2e6-4a07-b757-2c2b8caa13df,experiment_first,6a212b8fffef44b6da07367b,control,7,7,100.000000,50,11,4.545455


## Metric 3: Conversational Latency / Session Duration (SD)

### Theoretical Construct
Session Duration (SD) serves as an operational proxy for **Temporal Efficiency** and conversational flow within the user interface. 

In human-AI interaction modeling, the total time elapsed during a task can indicate two diametrically opposed cognitive states:
1. **Fluid Problem Solving:** A lower session duration combined with high task success implies high system usability, low cognitive load, and high temporal efficiency.
2. **Cognitive Overload or System Friction:** A high session duration can indicate that a user struggled with the interface, faced comprehension barriers, spent excessive time parsing AI responses, or was delayed by sluggish system latency.

### Mathematical Definition
For each participant session $i$ within an experimental condition, the Session Duration ($\text{SD}_i$) is calculated as the delta between the timestamp of the very last captured message (regardless of whether it was sent by the user or the AI) and the timestamp of the very first message initiating the session:

$$\text{SD}_{i} = t_{\text{last\_message}} - t_{\text{first\_message}}$$

The raw delta is transformed into a clean **decimal minute** format:
$$\text{SD}_{i} \text{ (minutes)} = \frac{\Delta t \text{ in seconds}}{60}$$

*Note: By utilizing both human and AI messages to define the boundaries, this metric accurately encapsulates the end-to-end user experience, including the system's final turnaround latency.*

In [8]:
# Convert the filtered messages list into a temporary DataFrame for vectorized calculations
df_msg_temp = pd.DataFrame(filtered_messages)

# Ensure the timestamp column is parsed as datetime objects
df_msg_temp["createdAt"] = pd.to_datetime(df_msg_temp["createdAt"])

# 2. Aggregate Min and Max timestamps per Chat Session
# This finds the exact start and end of the conversation
session_timestamps = df_msg_temp.groupby("chatId")["createdAt"].agg(["min", "max"]).reset_index()

# 3. Calculate Session Duration (SD) in Decimal Minutes
# (Timestamp difference converted to total seconds, then divided by 60)
session_timestamps["sd"] = (session_timestamps["max"] - session_timestamps["min"]).dt.total_seconds() / 60.0

# Rename columns for clarity before merging
session_timestamps = session_timestamps.rename(
    columns={
        "chatId": "chat_id",
        "min": "session_start_time",
        "max": "session_end_time"
    }
)

# Convert chat_id to string to match the main DataFrame's data type
session_timestamps["chat_id"] = session_timestamps["chat_id"].astype(str)

df_metrics = df_metrics.merge(session_timestamps[["chat_id", "session_start_time", "session_end_time", "sd"]], on="chat_id", how="left")

# Fill potential NaN values with 0 (e.g., if a session had 0 or 1 message total)
df_metrics["sd"] = df_metrics["sd"].fillna(0.0)

## Metric 4: System Assistance Need Score (SANS)

### Theoretical Construct
The System Assistance Need Score—historically conceptualized as the *Average Number of Hints*—serves as an operational proxy for **Task Independence** versus **Systemic Guidance Need**. 

In user experience scaffolding and human-AI interaction frameworks, a participant's inability to resolve a scenario autonomously indicates a misalignment between the user's mental model and the system's interface transparency. CAPS quantifies this friction through a weighted penalty system:
* **Hints** ($w=1$) indicate a minor breakdown in task clarity, where a subtle conceptual nudge is sufficient to restore autonomy.
* **Solutions** ($w=2$) represent a critical breakdown in usability or language understanding, requiring the system to expose the direct path to resolution and overriding active user exploration.

*Directional Interpretation:* **Lower is better.** A SANS score of $0$ denotes absolute task independence, whereas higher values indicate heavy systemic intervention and lower standalone usability.

### Mathematical Definition
For each participant session $i$ within an experimental condition, SANS is calculated as a weighted linear combination of the total number of hints and solutions requested by or provided to the user during that specific session:

$$\text{SANS}_{i} = (1 \times \text{Hints Requested}_{i}) + (2 \times \text{Solutions Requested}_{i})$$

*Note: Because this metric evaluates scaffolding reliance, it serves as a vital covariate when interpreting Task Success Rate (TSR) and Session Duration (SD).*

In [9]:
sans_data = []

for chat in filtered_chats:
    chat_id = str(chat.get("_id"))

    task_list = chat.get("taskList", [])

    hints_count = 0
    solutions_count = 0
    for task in task_list:

        hint = task.get("hint")
        if hint.get("used") == True: 
            hints_count += 1

        solution = task.get("solution")
        if solution.get("used") == True:
            solutions_count += 1
    
    # Calculate the weighted CAPS metric
    sans_score = (1 * hints_count) + (2 * solutions_count)
    
    sans_data.append({
        "chat_id": chat_id,
        "hints_used": hints_count,
        "solutions_used": solutions_count,
        "sans": sans_score
    })

df_caps = pd.DataFrame(sans_data)

# merge to master df 
df_metrics = df_metrics.merge(df_caps, on="chat_id", how="left")

### Data analysis

In [10]:
# 1. Define the structural columns (index) and the columns we want to reshape
index_cols = ['pid', 'uid', 'group']
metric_cols = ['tsr', 'aml', 'sd', 'sans']

# 2. Pivot the DataFrame
# 'columns="condition"' splits your metrics into separate columns based on 'control' or 'experiment'
df_wide = df_metrics.pivot(
    index=index_cols,
    columns='condition',
    values=metric_cols
)

# 3. Flatten the Multi-Level Column Index
# After pivoting, columns look like: ('tsr', 'control'), ('tsr', 'experiment')
# We map them to clean strings like 'tsr_ctrl' and 'tsr_exp'
df_wide.columns = [
    f"{metric}_{'ctrl' if cond == 'control' else 'exp'}" 
    for metric, cond in df_wide.columns
]

# 4. Reset the index to turn 'pid', 'uid', and 'group' back into regular columns
df_wide = df_wide.reset_index()

# 5. Validation Check
print("--- Transformation Summary ---")
print(f"Shape of wide DataFrame: {df_wide.shape} (Expected: 20 rows, 11 columns)")
print("\nGenerated Columns:")
print(list(df_wide.columns))
df_wide

--- Transformation Summary ---
Shape of wide DataFrame: (20, 11) (Expected: 20 rows, 11 columns)

Generated Columns:
['pid', 'uid', 'group', 'tsr_ctrl', 'tsr_exp', 'aml_ctrl', 'aml_exp', 'sd_ctrl', 'sd_exp', 'sans_ctrl', 'sans_exp']


,pid,uid,group,tsr_ctrl,tsr_exp,aml_ctrl,aml_exp,sd_ctrl,sd_exp,sans_ctrl,sans_exp
0,1,3d3fad0c-f4e2-491f-97b3-95c71b7c6723,control_first,100.000000,100.000000,4.777778,5.500000,7.422933,9.862233,6.0,6.0
1,2,700afe5f-e6dc-4191-9ffc-2d540d21780d,experiment_first,100.000000,100.000000,3.625000,6.000000,4.836250,6.980950,11.0,6.0
2,3,071b323f-15a1-4f26-8510-9a857540cf04,experiment_first,71.428571,71.428571,4.214286,4.285714,2.993833,2.978133,0.0,0.0
3,4,ecd6e48c-d160-4ad1-b576-772aee78a77a,control_first,100.000000,100.000000,7.000000,3.909091,7.622383,5.821300,5.0,5.0
4,5,f21ef8da-e2e6-4a07-b757-2c2b8caa13df,experiment_first,100.000000,100.000000,4.545455,6.555556,4.212067,2.528067,1.0,0.0
5,6,2dbd9cc6-7d1f-4b50-9dc4-f22ba6c4aa09,control_first,100.000000,85.714286,7.461538,4.076923,4.835250,3.759750,0.0,1.0
6,7,4ddbf84b-633f-49e1-9412-e52629d7eb1e,experiment_first,14.285714,71.428571,3.200000,3.352941,8.711367,10.146417,0.0,3.0
7,8,28ea6a5b-7ea9-41b2-851c-1317b0df0019,experiment_first,100.000000,100.000000,3.750000,3.500000,3.243350,4.329150,6.0,8.0
8,9,05e5bd34-f2d8-4a3d-9453-e9f413cdfcf7,control_first,100.000000,100.000000,8.571429,5.153846,3.064567,10.648217,0.0,0.0
9,10,cd83ed7e-57d0-4ec5-8210-5acbbe5accae,control_first,100.000000,100.000000,2.818182,2.909091,8.644917,7.812450,0.0,0.0


In [11]:
from typing import Literal

def analyze_metric(
        df: pd.DataFrame,
        metric: Literal["tsr", "aml", "sd", "sans"],
        metric_label: str,
        order_col="group"
):
    # Step 1: Extract paired columns based on your exact naming convention
    data_ctrl = df[f"{metric}_ctrl"].astype(float)
    data_exp = df[f"{metric}_exp"].astype(float)

    # Step 2: Calculate the personal difference (Experiment minus Control)
    diffs = data_exp - data_ctrl

    # Step 3: Run Shapiro-Wilk to test for normal distribution
    _, p_shap = st.shapiro(diffs)
    is_normal = p_shap > 0.05

    # Step 4: Run paired T-test
    ttest_res = pg.ttest(data_exp, data_ctrl, paired=True)
    p_ttest = float(ttest_res["p_val"].iloc[0])
    d_ttest = float(ttest_res["cohen_d"].iloc[0])

    # Step 5: Run Wilcoxon Signed-Rank test as backup
    wilc_res = pg.wilcoxon(data_exp, data_ctrl)
    p_wilc = float(wilc_res["p_val"].iloc[0])

    # Step 6: Clean up counterbalancing strings to prevent grouping bugs
    # FIXED: Replaced 'df_metrics' with your input variable 'df'
    order_series = df[order_col].astype(str).str.strip().str.lower()
    order_groups = order_series.unique()

    # Step 7: Calculate the Carry-Over effect (independent T-test on differences)
    if len(order_groups) == 2:
        diffs_g1 = diffs[order_series == order_groups[0]]
        diffs_g2 = diffs[order_series == order_groups[1]]

        carry_res = pg.ttest(diffs_g1, diffs_g2, paired=False)
        p_carry = float(carry_res["p_val"].iloc[0])
    else:
        p_carry = np.nan

    # Step 8: Dynamic routing based on the Shapiro-Wilk result
    # FIXED: Replaced the malformed code blocks with functioning pandas extractions
    if is_normal:
        t_stat = float(ttest_res["T"].iloc[0])
        stat_reported = f"t(19) = {t_stat:.2f}"
        p_reported = p_ttest
        es_reported = f"dz = {d_ttest:.2f}"
        test_type = "Parametric (t-test)"
    else:
        # Dynamically safety check output naming keys for Wilcoxon stats
        w_col = "W-val" if "W-val" in wilc_res.columns else "W_val"
        r_col = "RBC" if "RBC" in wilc_res.columns else "r"

        w_stat = float(wilc_res[w_col].iloc[0])
        r_rbc = float(wilc_res[r_col].iloc[0])

        stat_reported = f"W = {w_stat:.1f}"
        p_reported = p_wilc
        es_reported = f"r = {r_rbc:.2f}"
        test_type = "Non-Parametric (Wilcoxon)*"

    # Step 9: Format the p-value cleanly to fit official APA 7 guidelines
    p_clean = (
        "< .001" if p_reported < 0.001 else f"{p_reported:.3f}".lstrip("0")
    )
    p_carry_clean = (
        "< .001" if p_carry < 0.001 else f"{p_carry:.3f}".lstrip("0")
    )

    # Step 10: Pack everything into a clean summary dictionary
    return {
        "Metric": metric_label,
        "Mean Control (SD)": f"{data_ctrl.mean():.2f} ({data_ctrl.std():.2f})",
        "Mean Experiment (SD)": f"{data_exp.mean():.2f} ({data_exp.std():.2f})",
        "Mean Diff": f"{diffs.mean():.2f}",
        "Shapiro p": f"{p_shap:.3f}",
        "Test Stat": stat_reported,
        "p-value": p_clean,
        "Effect Size": es_reported,
        "Carry-Over p": p_carry_clean,
        "Method": test_type,
    }

task_metrics = [
    {"key": "tsr", "label": "Task Success Rate (%)"},
    {"key": "aml", "label": "Message Length (Words)"},
    {"key": "sd", "label": "Session Duration (Min)"},
    {"key": "sans", "label": "Assistance Score (SANS)"},
]

performance_rows = []
for m in task_metrics:
    row_output = analyze_metric(
        df=df_wide,
        metric=m["key"],
        metric_label=m["label"],
        order_col="group",
    )
    performance_rows.append(row_output)

# Display your publication-ready results
df_final_performance = pd.DataFrame(performance_rows)
print(
    df_final_performance[
        [
            "Metric",
            "Mean Control (SD)",
            "Mean Experiment (SD)",
            "Shapiro p",
            "Method",
            "Test Stat",
            "p-value",
            "Effect Size",
            "Carry-Over p",
        ]
    ].to_string(index=False)
)

df_final_performance[
    [
        "Metric",
        "Mean Control (SD)",
        "Mean Experiment (SD)",
        "Shapiro p",
        "Method",
        "Test Stat",
        "p-value",
        "Effect Size",
        "Carry-Over p",
    ]
].to_csv("./task_performance.csv", index=False, sep=';', encoding='utf-8')


                 Metric Mean Control (SD) Mean Experiment (SD) Shapiro p                     Method     Test Stat p-value Effect Size Carry-Over p
  Task Success Rate (%)     87.14 (28.15)         93.57 (9.80)     0.000 Non-Parametric (Wilcoxon)*      W = 10.0    .547    r = 0.29         .533
 Message Length (Words)       6.04 (2.61)          4.94 (1.67)     0.113        Parametric (t-test) t(19) = -2.18    .042   dz = 0.50         .033
 Session Duration (Min)       5.79 (2.60)          6.71 (4.02)     0.003 Non-Parametric (Wilcoxon)*      W = 80.0    .368    r = 0.24         .313
Assistance Score (SANS)       2.30 (3.37)          3.25 (4.51)     0.001 Non-Parametric (Wilcoxon)*      W = 13.5    .166    r = 0.51         .113
